In [1]:
import psycopg2
from psycopg2 import sql
from psycopg2.extras import execute_values, Json
import logging
import json
import html
import re

In [2]:
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

In [3]:
DB_PARAMS = {
    'dbname':   'thecall',
    'user':     'postgres',
    'password': 'password',
    'host':     'localhost',
    'port':     '5432'
}

TABLE_SCHEMA = 'articles'
TABLE_NAME   = 'filelist'

# Rows per commit. Reduce if you hit memory limits.
BATCH_SIZE = 500

In [4]:
# ── Database helpers ──────────────────────────────────────────────────────────

def connect_to_db(params):
    try:
        logging.info("Connecting to PostgreSQL...")
        conn = psycopg2.connect(**params)
        conn.autocommit = False
        logging.info("Connected.")
        return conn
    except psycopg2.Error as e:
        logging.error(f"Connection failed: {e}")
        return None



In [5]:
def export_unified_json(output_path):
    conn = connect_to_db(DB_PARAMS)
    if not conn:
        return

    query = sql.SQL("""
        SELECT json_unified 
        FROM {}.{} 
        WHERE nitearticle = 1
    """).format(
        sql.Identifier(TABLE_SCHEMA),
        sql.Identifier(TABLE_NAME)
    )

    articles = []
    try:
        with conn.cursor() as cur:
            cur.execute(query)
            # Fetch all rows; each row[0] is already a dict thanks to psycopg2
            rows = cur.fetchall()
            articles = [row[0] for row in rows if row[0] is not None]
            
        # Write to file using standard json.dump
        # ensure_ascii=False keeps your special characters (like curly quotes) as-is
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(articles, f, indent=2, ensure_ascii=False)
            
        logging.info(f"Successfully exported {len(articles)} articles to {output_path}")

    except Exception as e:
        logging.error(f"Export failed: {e}")
    finally:
        conn.close()

# Run the export
export_unified_json('E:/Callproject/nitearticles.json')

INFO: Connecting to PostgreSQL...
INFO: Connected.
INFO: Successfully exported 124 articles to E:/Callproject/nitearticles.json
